# Week 2 - Day 2: Vectorize Your Own Data (TF-IDF From Scratch)

In this notebook, we will take the `intents.csv` dataset built on Day 1 and run our own custom `tf()`, `idf()`, and `tfidf()` functions over it to analyze word weights across different intents.

## Part A — Custom TF-IDF Functions & Corpus Vectorization

In [ ]:
import csv
import re
import math
import os

def clean_and_tokenize(text):
    """Cleans punctuation and tokenizes text to lowercase tokens."""
    text = text.lower()
    return re.findall(r'\b\w+\b', text)

def tf(word, document_tokens):
    """Term Frequency: Count of word / total tokens in document"""
    if not document_tokens:
        return 0
    return document_tokens.count(word) / len(document_tokens)

def idf(word, all_documents_tokens):
    """Inverse Document Frequency: log(Total Docs / Docs containing word)"""
    N = len(all_documents_tokens)
    num_docs_with_word = sum(1 for doc in all_documents_tokens if word in doc)
    if num_docs_with_word == 0:
        return 0
    return math.log(N / num_docs_with_word)

def tfidf(word, document_tokens, all_documents_tokens):
    """Computes the final TF-IDF lexical weight."""
    return tf(word, document_tokens) * idf(word, all_documents_tokens)

### Load Dataset & Group Sentences by Intent
Every intent's collective sentences will be treated as a single document.

In [ ]:
# Create dummy intents.csv if it doesn't exist for the script to run seamlessly
if not os.path.exists('intents.csv'):
    dummy_data = [
        ["text", "label"],
        ["remind me to buy milk tomorrow", "create_task"],
        ["set an alarm for 7 am", "create_task"],
        ["make a note that i spent 20 dollars", "save_memory"],
        ["remember that my car keys are on the table", "save_memory"],
        ["what is the weather like today", "get_weather"],
        ["will it rain in pristina this weekend", "get_weather"],
        ["play some relaxing music on spotify", "play_media"],
        ["turn up the volume please", "play_media"],
        ["book a flight to tirana next week", "book_travel"],
        ["find a cheap hotel in saranda", "book_travel"],
        ["tell me a funny joke to make me laugh", "entertainment"],
        ["do you know any good riddles", "entertainment"]
    ]
    with open('intents.csv', mode='w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(dummy_data)

# Process the CSV
intents_data = {}
with open('intents.csv', mode='r', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader, None)  # Skip header
    for row in reader:
        if len(row) >= 2:
            text, label = row[0], row[1].strip()
            if label not in intents_data:
                intents_data[label] = []
            intents_data[label].append(text)

intent_names = list(intents_data.keys())
all_documents_tokens = []
intent_to_tokens = {}

for intent in intent_names:
    combined_text = " ".join(intents_data[intent])
    tokens = clean_and_tokenize(combined_text)
    intent_to_tokens[intent] = tokens
    all_documents_tokens.append(tokens)

### Generate Top-5 Terms Table Per Intent

In [ ]:
print(f"{'Intent':<18} | {'Top-5 Terms (Term -> Weight)':<50}")
print("-" * 75)
for intent in intent_names:
    current_tokens = intent_to_tokens[intent]
    unique_words = set(current_tokens)
    
    word_scores = {}
    for word in unique_words:
        word_scores[word] = tfidf(word, current_tokens, all_documents_tokens)
        
    sorted_words = sorted(word_scores.items(), key=lambda x: x[1], reverse=True)
    top_5 = sorted_words[:5]
    
    formatted_terms = ", ".join([f"{w} ({s:.3f})" for w, s in top_5])
    print(f"{intent:<18} | {formatted_terms}")

## Part B — Find the Confusions

**Shared Word Confusion Risk Cases:**

1. **Word:** `"me"` / `"my"` 
   * *Explanation:* The word ranks high across both `create_task` and `save_memory` intents (e.g., "remind me", "remember my"), making it a strong confusion risk because the classifier might evaluate intents based heavily on pronouns rather than distinguishing verbs.
2. **Word:** `"to"` 
   * *Explanation:* This preposition appears with high lexical frequency when setting tasks ("to buy") and booking travel ("to tirana"), causing potential overlaps if text inputs are short.

---

## Reflection

**Lexical Limits of TF-IDF:**
TF-IDF will entirely fail on a sentence using semantic synonyms like *"purchased"* if the training corpus only contains the token *"buy"* or *"bought"*. Because it evaluates features purely on exact token matching (lexical), it possesses zero contextual or semantic understanding of identical sentence meanings.